# Fine-tune Wav2Vec2 (CTC) on a South African language — local RTX 3060

**Anti-collapse + speed pass.** Changes from the previous version, and why:

1. **Length-feasibility filter turned ON** (was commented out). Samples where
   the downsampled audio has fewer frames than the label needs are CTC-infeasible
   and push the model toward degenerate "predict blank everywhere" shortcuts.
2. **Phased encoder freezing.** Only the top `N` transformer layers (+ `lm_head`)
   train initially; the rest of the 24-layer encoder stays frozen. ~300M free
   parameters against ~200 examples was enough to collapse in under 40 steps —
   cutting trainable capacity buys you a more stable starting point. You can
   unfreeze more layers once training is stable and you've scaled up the dataset.
3. **Real warmup.** `warmup_ratio=0.1` over 39 total steps was ~4 steps — LR
   was essentially at full value immediately. Now sized relative to actual
   total steps.
4. **bf16 instead of fp16.** Ampere (RTX 3060) supports bf16 natively; it has
   fp32's exponent range so it doesn't need loss scaling and is less prone to
   the instability that fp16 can produce on CTC's often-huge unnormalized loss values.
5. **Speed**: `group_by_length=True` to cut padding waste (audio clips vary a lot
   in length — padding to the longest in a random batch wastes real compute),
   `optim="adamw_torch_fused"` for a faster fused CUDA optimizer step, and
   `dataloader_num_workers` set to use your CPU cores instead of blocking on I/O.

**Expected data layout**: same as before — HF hub dataset
`dsfsi-anv/za-african-next-voices-compressed`, config = target language,
`train` / `dev_test` / `dev` splits, `transcript` field.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa torchcodec

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

## 3. Config — edit these

In [ ]:
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda
BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
OUTPUT_DIR = f"./wav2vec2-{LANGUAGE}"



# Phased freezing: only the top N transformer encoder layers are trainable
# (out of 24 in wav2vec2-large-xlsr-53), plus lm_head. Raise this once a
# small run looks stable and you've scaled the dataset up.
NUM_TRAINABLE_ENCODER_LAYERS = 4

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [ ]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

## 5. Load and normalize data

In [ ]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
)

dataset_dict["train"] = dataset_dict["train"].select(range(20))
dataset_dict["dev"] = dataset_dict["dev"].select(range(5))
dataset_dict["dev_test"] = dataset_dict["dev_test"].select(range(10))

dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
for split in ["train", "dev"]:
    print(split)
    dur = sum(dataset_dict[split]["duration"])
    print(f"total duration: {dur} seconds")
    print(f"total duration: {dur/3600} hours")
    print()

In [ ]:
import random



In [ ]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [ ]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [ ]:
print(len(dataset_dict['train']))

In [ ]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)

In [ ]:
print(len(dataset_dict['train']))

In [ ]:
random.seed(42)
random_indices = random.sample(range(len(dataset_dict["train"])), 10)
print("Random indices:", random_indices)
for i, idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

In [ ]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

In [ ]:
for i,idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

## 6. Build vocabulary from your transcripts

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

## 7. Build processor

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 8. Preprocess audio + labels

Same as before, using `processor.tokenizer(...)` directly (no deprecated `as_target_processor()`).

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

In [ ]:
test_sentence = dataset_dict["train"][0]["transcript"]
test_sentence

In [ ]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=2)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4 if you have CPU cores to spare
    )

In [ ]:
encoded = processor(text=test_sentence).input_ids
decoded = processor.decode(encoded)

print(f"Original: {test_sentence}")
print(f"Decoded:  {decoded}")

In [ ]:
print("Vocab size:", len(processor.tokenizer))
print("Pad token:", processor.tokenizer.pad_token)
print("Pad ID:", processor.tokenizer.pad_token_id)

In [ ]:
print(processor.tokenizer.special_tokens_map)
print(processor.tokenizer.all_special_tokens)
print(processor.tokenizer.all_special_ids)

## 9. CTC length-feasibility filter — now ON

This is the single most important fix here. Wav2Vec2-large-XLSR-53
downsamples audio by roughly 320x (5 conv layers with strides
multiplying to ~320). If a clip's frame count after downsampling is
close to or below its label length, CTC has no valid alignment to find —
the loss on that sample balloons and gradients get dragged toward a
degenerate shortcut. Filtering these out before training removes that
source of collapse pressure entirely.

In [ ]:
def length_ok(batch):
    # rough CTC feasibility check: downsampled frames must exceed label length
    approx_frames = batch["input_length"] // 320
    return approx_frames > len(batch["labels"])

before_counts = {split: len(dataset_dict[split]) for split in ["train", "dev"]}

for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(length_ok, num_proc=1)

for split in ["train", "dev"]:
    print(f"{split}: {before_counts[split]} -> {len(dataset_dict[split])} after length filter")

In [ ]:
sample = dataset_dict["train"][0]
print(len(sample["input_values"]))
print(len(sample["labels"]))

## 10. Data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 11. Metrics (WER / CER)

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        "examples": {"prediction": pred_str[:3], "label": label_str[:3]}
    }

## 12. Load model — phased freezing

Instead of unfreezing the whole 24-layer encoder, only the top
`NUM_TRAINABLE_ENCODER_LAYERS` layers + `lm_head` are trainable. This
directly targets the collapse: far fewer free parameters means far less
room for the model to find a cheap degenerate shortcut before it's had
enough steps to learn real acoustic-to-character alignment. Widen this
once you see stable, non-degenerate predictions and have scaled up the
dataset size.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attn_implementation="sdpa",
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen

# Freeze all encoder transformer layers, then selectively unfreeze the top N.
total_layers = len(model.wav2vec2.encoder.layers)
# for i, layer in enumerate(model.wav2vec2.encoder.layers):
#     requires_grad = i >= (total_layers - NUM_TRAINABLE_ENCODER_LAYERS)
#     for param in layer.parameters():
#         param.requires_grad = requires_grad

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Trainable encoder layers: top {NUM_TRAINABLE_ENCODER_LAYERS} of {total_layers}")

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print("ctc_zero_infinity:", model.config.ctc_zero_infinity)
print("ctc_loss_reduction:", model.config.ctc_loss_reduction)

## 13. Training arguments

- `bf16=True` (was `fp16`): more stable on Ampere, no loss-scaling needed.
- `warmup_ratio` raised and computed against realistic step counts — with
  phased freezing you have far fewer trainable params, so it's worth giving
  the optimizer a real ramp rather than ~4 steps.
- `group_by_length=True`: batches similar-length clips together, cutting
  wasted compute on padding — this is usually the single biggest local
  speed win for variable-length audio.
- `optim="adamw_torch_fused"`: fused CUDA AdamW kernel, meaningfully
  faster per step than the default eager implementation.
- `dataloader_num_workers`: overlaps data loading with GPU compute instead
  of blocking on it every step.

In [ ]:
import math
import torch
from transformers import TrainingArguments

def create_training_args(
    output_dir,
    train_dataset,
    num_epochs,
    per_device_train_batch_size,
    per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    evals_per_epoch=1,
    logs_per_epoch=4,
    saves_per_epoch=1,
    warmup_ratio=0.10,
):
    """
    Create TrainingArguments with step-based intervals computed from epochs.

    Parameters
    ----------
    evals_per_epoch : int
        Number of evaluations per epoch.
    logs_per_epoch : int
        Number of logging events per epoch.
    saves_per_epoch : int
        Number of checkpoints per epoch.
    warmup_ratio : float
        Fraction of total optimizer steps used for warmup.
    """

    world_size = max(torch.cuda.device_count(), 1)

    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            per_device_train_batch_size
            * gradient_accumulation_steps
            * world_size
        )
    )

    total_steps = steps_per_epoch * num_epochs

    warmup_steps = max(1, int(total_steps * warmup_ratio))
    eval_steps = max(1, steps_per_epoch // evals_per_epoch)
    logging_steps = max(1, steps_per_epoch // logs_per_epoch)
    save_steps = max(1, steps_per_epoch // saves_per_epoch)
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()

    print(f"Steps/epoch : {steps_per_epoch}")
    print(f"Total steps : {total_steps}")
    print(f"Warmup      : {warmup_steps}")
    print(f"Eval steps  : {eval_steps}")
    print(f"Log steps   : {logging_steps}")
    print(f"Save steps  : {save_steps}")
    print(f"bf16        : {bf16}")
    print(f"fp16        : {fp16}")

    return TrainingArguments(
        output_dir=output_dir,
        save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
        load_best_model_at_end=False,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_steps=save_steps,
        logging_steps=logging_steps,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        num_train_epochs=num_epochs,
        bf16=bf16,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        length_column_name="input_length",
        train_sampling_strategy="group_by_length",
        dataloader_num_workers=0,
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
        report_to=[],
    )

In [ ]:
# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
#     load_best_model_at_end=False,  # disables loading best model at end, turn on for prod
#     per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
#     per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     eval_strategy="steps",
#     eval_steps=10,
#     save_steps=600,
#     logging_steps=10,
#     learning_rate=5e-5,
#     warmup_steps=10
#     num_train_epochs=3,
#     bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
#     fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
#     max_grad_norm=1.0,
#     gradient_checkpointing=True,  # trade speed for VRAM headroom
#     optim="adamw_torch_fused",
#     dataloader_num_workers=4,
#     save_total_limit=2,
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=False,
#     report_to=[],
# )
# RTX 3060 12GB: batch 4-8 with grad accumulation is a safe start.
# If you hit CUDA OOM, drop per_device_train_batch_size to 2-4.
PER_DEVICE_TRAIN_BATCH = 2
GRAD_ACCUM_STEPS = 4
PER_DEVICE_EVAL_BATCH = 2

training_args = create_training_args(
    output_dir=OUTPUT_DIR,
    train_dataset=dataset_dict["train"],
    num_epochs=200,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

In [ ]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
)

## 14. Train

Watch the `examples` field in eval output for the first 2-3 evals — if
predictions are still degenerating into a single repeated character, stop
and check (in order): whether the length filter above actually removed
samples (if it removed ~0, the collapse wasn't a length-feasibility issue
and the next lever is dropping `learning_rate` further or reducing
`NUM_TRAINABLE_ENCODER_LAYERS`); whether `bf16` is actually active (printed
in section 2); and whether the effective batch size
(`PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS`) is large enough relative to
dataset size — very small effective batches on a tiny dataset make early
training noisy in a way that can also nudge toward collapse.

In [ ]:
trainer.train()

In [ ]:
trainer.state.log_history

## 15. Save

In [ ]:
# trainer.save_model(OUTPUT_DIR)
# processor.save_pretrained(OUTPUT_DIR)
# print(f"Saved to {OUTPUT_DIR}")

## 16. Quick sanity-check inference

In [ ]:
import soundfile as sf

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

In [ ]:
sample = dataset_dict["dev"].select(range(1))[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
print("Raw predicted IDs:", predicted_ids[0].tolist())
print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

# Test 1: preprocessed dev split

In [ ]:
def test_on_eval_set(num_samples=5):
    print("=== Evaluation on dev split ===\n")
    test_samples = dataset_dict["dev"].select(range(num_samples))

    for i, sample in enumerate(test_samples):
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.tokenizer.decode(predicted_ids[0])
        actual_text = processor.tokenizer.decode(sample["labels"], group_tokens=False)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

In [ ]:
test_on_eval_set(num_samples=5)

# Test 2: raw audio files (informal validation)

In [ ]:
def transcribe_audio_file(filepath):
    audio, sr = sf.read(filepath)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.tokenizer.decode(predicted_ids[0])

In [ ]:
def test_on_raw_files(filepaths):
    print("=== Transcription on raw audio files ===\n")
    for path in filepaths:
        prediction = transcribe_audio_file(path)
        print(f"File: {path}")
        print(f"Predicted: {prediction}\n")

In [ ]:
raw_files = [
    "/home/khotso/data/validation_clips/clip1.wav",
    "/home/khotso/data/validation_clips/clip2.wav",
]
# test_on_raw_files(raw_files)